# 06 · Datos monetarios: Censos Económicos 2024 y ENIGH 2024

**Objetivo:** bajar las dos fuentes de dinero que usa la extensión del proyecto.

| Fuente | Qué aporta | Nivel | Año de los datos |
|---|---|---|---|
| **Censos Económicos 2024** | Ingresos, gastos, sueldos y personal de los restaurantes | Municipio × clase SCIAN | 2023 |
| **ENIGH 2024** (microdatos) | Cuánto gastan los hogares en comer fuera de casa | Hogar | 2024 |

- Con los **Censos Económicos** se estima cuánto vende y cuánto gana un restaurante según su tipo, su municipio y su tamaño (oferta, notebook 07).
- Con la **ENIGH** se estima cuánto dinero gastan en comer fuera los hogares y los trabajadores de cada zona (demanda, notebook 08).

Ningún dato público da las ventas de cada restaurante: los Censos Económicos publican totales por municipio y actividad. Por eso todo lo que sigue son **estimaciones**, no ventas observadas.

Las ligas se tomaron de las páginas de descarga del INEGI. Si alguna cambia, búscala en la sección *Datos abiertos* (Censos Económicos) o *Microdatos* (ENIGH) y reemplázala abajo.

## 0. Librerías y rutas

In [ ]:
from pathlib import Path
import io, tempfile, time, zipfile, urllib.request
import pandas as pd

ROOT = Path.cwd().parent
RAW, PROC = ROOT / "data/raw", ROOT / "data/processed"
RAW.mkdir(parents=True, exist_ok=True)

URLS = {
    "ce_qro_2024": "https://www.inegi.org.mx/contenidos/programas/ce/2024/datosabiertos/conjunto_de_datos_ce_qro_2024_csv.zip",
    "enigh_concentradohogar": "https://www.inegi.org.mx/contenidos/programas/enigh/nc/2024/microdatos/enigh2024_ns_concentradohogar_csv.zip",
    "enigh_hogares": "https://www.inegi.org.mx/contenidos/programas/enigh/nc/2024/microdatos/enigh2024_ns_hogares_csv.zip",
}
MUNICIPIOS_ZMQ = {6: "Corregidora", 8: "Huimilpan", 11: "El Marqués", 14: "Querétaro"}

## 1. Descargar

In [ ]:
def descargar(url, destino, intentos=3):
    if destino.exists():
        print(f"Ya existe: {destino.name}")
        return destino
    for i in range(intentos):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=600) as r, open(destino, "wb") as f:
                while bloque := r.read(1 << 20):
                    f.write(bloque)
            print(f"Descargado: {destino.name} ({destino.stat().st_size / 1e6:.1f} MB)")
            return destino
        except Exception as e:
            print(f"  intento {i + 1} falló: {e}")
            time.sleep(5)
    raise RuntimeError(f"No se pudo descargar {url}")

zips = {k: descargar(u, RAW / u.split("/")[-1]) for k, u in URLS.items()}

## 2. Leer un CSV dentro de un zip

Cada zip trae el conjunto de datos y otros archivos (diccionario, catálogos, metadatos). Se lee el CSV más grande, que es el conjunto de datos. Las columnas se pasan a minúsculas para no depender de cómo las escriba el INEGI.

In [ ]:
def leer_zip(ruta, **kw):
    with zipfile.ZipFile(ruta) as z:
        csvs = [i for i in z.infolist() if i.filename.lower().endswith(".csv")]
        info = max(csvs, key=lambda i: i.file_size)
        crudo = z.read(info)
    for enc in ("utf-8", "latin-1"):
        try:
            df = pd.read_csv(io.BytesIO(crudo), encoding=enc, dtype=str, low_memory=False, **kw)
            break
        except UnicodeDecodeError:
            continue
    df.columns = df.columns.str.strip().str.lower().str.replace('"', "")
    print(f"{ruta.name} → {info.filename}: {len(df):,} filas × {df.shape[1]} columnas")
    return df

ce = leer_zip(zips["ce_qro_2024"])
print("Columnas:", list(ce.columns)[:40])
ce.head()

## 3. Censos Económicos: quedarse con los restaurantes de la ZM

Columnas que se usan (códigos del INEGI):

| Código | Qué es | Unidad |
|---|---|---|
| `ue` | Unidades económicas (establecimientos) | número |
| `h001a` | Personal ocupado total | personas |
| `m000a` | Ingresos por suministro de bienes y servicios | millones de pesos |
| `k000a` | Gastos por consumo de bienes y servicios | millones de pesos |
| `j000a` | Remuneraciones (sueldos) | millones de pesos |

Se guardan las filas del subsector 722 (incluidas sus clases de 6 dígitos) para los 4 municipios y para el total estatal, que sirve de respaldo cuando un municipio está oculto por confidencialidad. Las filas por estrato de tamaño se quitan: se usan los totales.

In [ ]:
def columna(df, *opciones):
    for o in opciones:
        if o in df.columns:
            return o
    raise KeyError(f"No encontré ninguna de {opciones}. Columnas: {list(df.columns)}")

c_mun = columna(ce, "e04", "municipio", "cve_mun", "mun")   # e04 = clave de municipio en los Censos Económicos
c_cod = columna(ce, "codigo", "codigo_scian", "actividad", "scian")
c_est = next((c for c in ("id_estrato", "estrato") if c in ce.columns), None)

ce2 = ce.copy()
ce2["codigo"] = ce2[c_cod].astype(str).str.strip().str.replace(" ", "")
ce2["cve_mun"] = pd.to_numeric(ce2[c_mun].astype(str).str.strip().str[-3:], errors="coerce").replace(0, pd.NA)   # vacío o 000 = total estatal
if c_est:
    ce2 = ce2[ce2[c_est].isna() | (ce2[c_est].astype(str).str.strip().isin(["", "0", "nan"]))]
ce2 = ce2[ce2["codigo"].str.startswith("722")]
ce2 = ce2[ce2["cve_mun"].isin(MUNICIPIOS_ZMQ) | ce2["cve_mun"].isna()]      # vacío = total estatal
VARS_CE = ["ue", "h001a", "m000a", "k000a", "j000a"]
faltan = [v for v in VARS_CE if v not in ce2.columns]
if faltan:
    raise KeyError(f"Faltan columnas en los Censos Económicos: {faltan}")
for v in VARS_CE:
    ce2[v] = pd.to_numeric(ce2[v].astype(str).str.replace(",", "").str.strip(), errors="coerce")
ce2 = ce2[["cve_mun", "codigo"] + VARS_CE].reset_index(drop=True)
ce2.to_parquet(PROC / "ce2024_restaurantes.parquet", index=False)
print(f"Filas de restaurantes: {len(ce2):,} · ocultas por confidencialidad (sin ingresos): {ce2['m000a'].isna().sum():,}")
ce2.sort_values(["cve_mun", "codigo"]).head(30)

## 4. ENIGH 2024: hogares urbanos

Se toman los hogares de localidades de **100 mil habitantes o más** (`tam_loc = 1`), que se parecen a la zona metropolitana. Se juntan dos tablas por `folioviv` + `foliohog`:

| Tabla | Columnas |
|---|---|
| `concentradohogar` | `ali_fuera` (gasto trimestral en alimentos fuera de casa), `tot_integ`, `ocupados`, `educa_jefe`, `ing_cor`, `ubica_geo`, `factor` |
| `hogares` | `num_auto` (automóviles del hogar) |

In [ ]:
conc = leer_zip(zips["enigh_concentradohogar"])
hog = leer_zip(zips["enigh_hogares"])

COLS_CONC = ["folioviv", "foliohog", "ubica_geo", "tam_loc", "educa_jefe", "tot_integ", "ocupados", "ali_fuera", "ing_cor", "factor"]
faltan = [c for c in COLS_CONC if c not in conc.columns]
if faltan:
    raise KeyError(f"Faltan columnas en concentradohogar: {faltan}")
enigh = conc[COLS_CONC].copy()
if "num_auto" in hog.columns:
    enigh = enigh.merge(hog[["folioviv", "foliohog", "num_auto"]], on=["folioviv", "foliohog"], how="left")
else:
    print("Aviso: la tabla hogares no trae num_auto; el modelo de demanda se ajustará sin automóviles.")

for c in enigh.columns.drop(["folioviv", "foliohog", "ubica_geo"]):
    enigh[c] = pd.to_numeric(enigh[c], errors="coerce")
if "entidad" in conc.columns:
    enigh["entidad"] = pd.to_numeric(conc["entidad"], errors="coerce")
else:
    enigh["entidad"] = pd.to_numeric(enigh["ubica_geo"].astype(str).str.zfill(5).str[:2], errors="coerce")
enigh = enigh[enigh["tam_loc"] == 1].reset_index(drop=True)
enigh.to_parquet(PROC / "enigh2024_hogares_urbanos.parquet", index=False)
print(f"Hogares urbanos (100 mil+): {len(enigh):,} · de Querétaro: {(enigh['entidad'] == 22).sum():,}")
enigh.describe().T.round(1)